# Text & Label Analysis — Results Overview

**Group 3 · Sponsor: Sit Stay Forever (SSF)** — reads the pipeline's output artifacts
(`outputs/csv/*.csv`) and reproduces the headline analyses. Pure pandas + matplotlib:
runs in either project environment, no OCR models needed.

Pipeline that produced these artifacts:
`ingest → Stage 1 PaddleOCR → Stage 2 Donut → Stage 3 thumbnail readability → Stage 4 linkage`
(plus `src/augment` robustness and `src/viz` figures).

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = ROOT / "outputs" / "csv"

s1 = pd.read_csv(CSV / "stage1_ocr.csv")
s1_img = pd.read_csv(CSV / "stage1_ocr_per_image.csv")
s3 = pd.read_csv(CSV / "stage3_readability.csv")
s4 = pd.read_csv(CSV / "stage4_linkage_summary.csv")
kw = pd.read_csv(CSV / "stage4_keyword_hits.csv")
rob = pd.read_csv(CSV / "augment_robustness.csv")
comp = pd.read_csv(CSV / "stage4_compliance_flags.csv")
print(f"Stage1: {len(s1)} text regions over {s1['image_id'].nunique()} images")
print(f"Stage3: {len(s3)} images | Stage4 joined: {s4.shape} | keyword hits: {len(kw)}")

## 1. What PaddleOCR reads on the sponsor's pack (sample)

In [ ]:
ssf_ids = s4.loc[s4["performance_tier"] == "sponsor", "image_id"]
(s1[s1["image_id"].isin(ssf_ids)][["image_id", "text", "rec_score"]]
   .sort_values(["image_id", "rec_score"], ascending=[True, False]).head(15))

## 2. Headline: the mobile-thumbnail cliff

Character retention = OCR characters recovered at the downsampled size / full resolution
(aspect-preserving downscale, no upscaling).

In [ ]:
tier = s4.groupby("performance_tier")[
    ["full_chars", "mobile_thumb_char_retention", "search_grid_char_retention"]
].mean().round(3)
print(tier)
print(f"\nOverall mobile retention: {s4['mobile_thumb_char_retention'].mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
order = ["sponsor", "high", "medium"]
colors = {"sponsor": "#2a78d6", "high": "#1baf7a", "medium": "#eda100"}
for t in order:
    sub = s4[s4["performance_tier"] == t]
    ys = [1.0, sub["search_grid_char_retention"].mean(), sub["mobile_thumb_char_retention"].mean()]
    ax.plot([0, 1, 2], ys, marker="o", color=colors[t], label=t)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["full", "320 px", "160 px"])
ax.set_ylabel("character retention"); ax.legend(); ax.set_ylim(0, 1.05)
ax.set_title("Text retention vs. serving resolution")
plt.tight_layout(); plt.show()

## 3. Keywords: what survives on mobile

Of the SSF search keywords visually present on-pack, how many still match in the
160 px thumbnail OCR?

In [ ]:
ssf_kw = (kw[kw["performance_tier"] == "sponsor"]
          .groupby("keyword")["survives_mobile_thumb"].any())
print(f"SSF keywords on-pack: {len(ssf_kw)} | survive mobile: {int(ssf_kw.sum())}")
ssf_kw.sort_values()

## 4. Robustness: photo conditions vs. resolution

Augmented variants (rotation, brightness, noise+blur — text-safe transforms, flips
excluded by design) barely dent OCR; the thumbnail size dominates.

In [ ]:
per = (rob[rob["transform"] != "original"]
       .groupby("transform")[["char_retention", "mean_score"]].mean().round(3))
per.loc["<mobile_thumb_160px>"] = [s4["mobile_thumb_char_retention"].mean().round(3), None]
per.sort_values("char_retention", ascending=False)

## 5. Main-image compliance screen

In [ ]:
comp[["original_name", "brand", "total_chars", "white_bg_compliance", "compliance_flags"]]

## 6. Validation vs. the sponsor dataset

Our OCR character counts against the dataset's independently computed `ocr_word_count`.

In [ ]:
sub = s4[["total_chars", "ocr_word_count"]].dropna()
print(f"corr(total_chars, ocr_word_count) = {sub['total_chars'].corr(sub['ocr_word_count']):.3f}  (n={len(sub)})")